# CIFAR-10 image classification model

## 1. Import libraries

In [ ]:
#Check tensorflow and numpy version
import tensorflow as tf
print(tf.__version__)
import numpy as np
print(np.__version__)

### TensorFlow version
The course boards run **TensorFlow 2.11** (CPU inference, step 2), and the Vitis AI 2.5 tools (step 3a) need a Keras 2 `.h5` model. Train with TensorFlow 2.11, which needs **Python 3.7 to 3.10**. If the cell above shows another TensorFlow version and your Python is 3.10 or older, uncomment and run the next cell, then restart the kernel.

Models saved by TensorFlow 2.16 or newer (Keras 3), for example on Google Colab, cannot be loaded by TensorFlow 2.11 or by Vitis AI 2.5.

In [ ]:
# %pip install "tensorflow==2.11.0" "numpy<2.0"   # only with Python <= 3.10, if the version above is not 2.11

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 2. Load the CIFAR-10 data

In [ ]:
# Load CIFAR-10 Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Original shapes:")
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)


def rgb_to_gray(x):
    """uint8 RGB images (..., 32, 32, 3) -> uint8 grayscale (..., 32, 32, 1).
    Integer arithmetic, so it gives exactly the same result in every step of the project
    (training here, quantization on the server, CPU and DPU inference on the Kria). Keep it identical everywhere.
    It is up to you to use RGB or grayscale input; this baseline uses grayscale."""
    x = x.astype(np.uint32)
    gray = (x[..., 0] * 299 + x[..., 1] * 587 + x[..., 2] * 114 + 500) // 1000
    return gray.astype(np.uint8)[..., np.newaxis]


x_train_gray = rgb_to_gray(x_train)
x_test_gray = rgb_to_gray(x_test)

# Normalize pixel values to [0, 1]
x_train_gray = x_train_gray.astype('float32') / 255.0
x_test_gray = x_test_gray.astype('float32') / 255.0

# Class names
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

num_classes = 10


## 3. Visualize the data

In [ ]:
# Visualize one sample per class
plt.figure(figsize=(12, 6))
for i in range(num_classes):
    idx = np.where(y_train.flatten() == i)[0][0]
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train_gray[idx].squeeze(), cmap='gray')
    plt.title(class_names[i])
    plt.axis('off')
plt.suptitle("Example Images for Each Class (Grayscale)")
plt.show()

# Show class distribution (training set only)
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 5))
plt.bar(class_names, counts)
plt.title("Training Data Class Distribution")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.show()


## 4. Build the model (an example CNN)

In [ ]:
model = models.Sequential([
    layers.Conv2D(64, (3,3), activation='relu', input_shape=(32,32,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'), 
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'), 
    layers.Flatten(),
    layers.Dense(256, activation='relu'), 
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 5. Train the model

In [ ]:
history = model.fit(
    x_train_gray, y_train,
    epochs=11,
    batch_size=128,
    validation_split=0.1, # 5000 images are used for validation
    verbose=1
)


# Plot Training & Validation Accuracy / Loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)
# Plot Accuracy
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs, acc, 'b-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r--', label='Validation Accuracy')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
# Plot Loss
plt.subplot(1,2,2)
plt.plot(epochs, loss, 'b-', label='Training Loss')
plt.plot(epochs, val_loss, 'r--', label='Validation Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


## 6. Evaluate the model on the test set

In [ ]:
#Evaluate the model on test data
test_loss, test_acc = model.evaluate(x_test_gray, y_test, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

#Confusion Matrix
y_pred = np.argmax(model.predict(x_test_gray), axis=1)
y_true = y_test.flatten()

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
plt.figure(figsize=(10,10))
disp.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.title("Confusion Matrix on Test Data")
plt.show()

## 7. Save the model
The file name is used in the next steps (quantization on the server, CPU inference on the Kria).

In [ ]:
model.save('cifar10_bw_model.h5')